# Snowball v8 — Notebook A：台股 Point-in-Time 自動化資料引擎

**定位（階段 2 + 階段 4 資料引擎）**：
- 每週五盤後／每月 11 日後執行一次（CPU 即可，不需 GPU）。
- 自動建置 **台股 1,000 檔 Point-in-Time 完整面板**：
  1. Yahoo Finance 歷史日價量（可調 `PERIOD = "8y"` 或 `"15y"`）＋ 滯後一期總經狀態；
  2. TWSE / TPEx 官方月估值快照（本益比、淨值比、殖利率）；
  3. **MOPS 官方上市/上櫃歷史月營收**（嚴格次月 11 日生效，計算近 3 月累計 YoY、加速度、創 12 月新高）；
  4. **George & Hwang (2004) 52 週新高距離**（嚴格因果）。
- 輸出壓縮版 `snowball_tw_1000_pit_v8.parquet`，供 **Notebook B（模型實驗室）** 直接掛載秒級訓練。

In [ ]:
# === Cell 1: 拉取最新 Snowball 引擎至 /tmp（保持 /kaggle/working 乾淨） ===
import subprocess
import sys
from pathlib import Path
import pandas as pd

REPO_URL = "https://github.com/948787878787/snowball.git"
LOCAL_SCRIPTS = Path("skills/snowball/scripts")
KAGGLE_REPO = Path("/tmp/snowball_repo")

if LOCAL_SCRIPTS.exists():
    SCRIPTS_DIR = LOCAL_SCRIPTS.resolve()
else:
    if not KAGGLE_REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(KAGGLE_REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(KAGGLE_REPO), "pull", "--ff-only"], check=False)
    SCRIPTS_DIR = KAGGLE_REPO / "skills/snowball/scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import build_training_panel as btp
import fetch_pit_fundamentals as fpf
import tw_monthly_rev as tmr

PERIOD = "8y"  # 可改為 "15y" 拉長跨度至 2011-2026（注意早期倖存者偏誤）
OUT_DIR = Path("/kaggle/working/snowball_data_v8") if Path("/kaggle/working").exists() else Path("./kaggle_dataset")
CACHE_DIR = Path("/tmp/pit_cache")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Engine ready from: {SCRIPTS_DIR} | Target period: {PERIOD}")

In [ ]:
# === Cell 2: 建置台股價量面板 + 合併 TWSE/TPEx 歷史估值 + MOPS 月營收 + 52 週新高 ===
panel = btp.build_combined_panel(markets=("TW",), period=PERIOD, horizons=(4, 12))
panel["date"] = pd.to_datetime(panel["date"]).astype("datetime64[ns]")
tw_tickers = sorted(panel["ticker"].unique().tolist())
start_y = int(panel["date"].min().year) - 1
end_y = int(panel["date"].max().year)

# 1. TWSE / TPEx 月估值 PIT
start_s = (panel["date"].min() - pd.Timedelta(days=60)).strftime("%Y-%m-%d")
end_s = panel["date"].max().strftime("%Y-%m-%d")
pit_tw = fpf.fetch_tw_exchange_pit(tw_tickers, cache_dir=CACHE_DIR, start_date=start_s, end_date=end_s)
if not pit_tw.empty:
    panel = fpf.merge_pit_into_panel(panel, pit_tw, max_staleness_days=200)

# 2. MOPS 官方歷史月營收（嚴格次月 11 日生效）
rev_raw = tmr.fetch_mops_monthly_revenue(start_y, end_y, cache_dir=CACHE_DIR, tickers_filter=tw_tickers, max_workers=8)
if not rev_raw.empty:
    rev_feat = tmr.compute_tw_monthly_rev_features(rev_raw)
    panel = tmr.merge_tw_monthly_rev_into_panel(panel, rev_feat, max_staleness_days=65)

# 3. George & Hwang (2004) 52 週新高距離
panel = tmr.add_52w_high_proximity(panel, price_col="close_usd")

out_path = OUT_DIR / "snowball_tw_1000_pit_v8.parquet"
panel.to_parquet(out_path, index=False, compression="zstd", compression_level=9)
print(f"Saved {out_path} ({out_path.stat().st_size / (1024*1024):.2f} MB) | shape={panel.shape}")
for col in ("pit_ep_z", "pit_roe_z", "pit_tw_rev_3m_yoy_z", "pit_tw_rev_accel_z", "pit_tw_rev_12m_high_z", "f_52w_high"):
    if col in panel.columns:
        print(f"  Coverage {col:24s}: {panel[col].notna().mean():.1%}")